# HealthBrowse Medical Model Lab — separate QLoRA experiment

This notebook develops a **separate adapter** for General and Clinicians. It does not change the live HealthBrowse model or call its API.

Start with **Runtime → Change runtime type → GPU**. You need access to the gated Meta Llama 3 repository and a Hugging Face read token saved in Colab Secrets under `HF_TOKEN` with notebook access enabled. Do not paste a token into code.

The default is a **one-step smoke test** using eight bundled, unreviewed exact-excerpt fixtures. Its weights are not a usable medical model. For a candidate run, first curate and review training and validation files using `healthbrowse_model_lab/DATASET.md`. No clinical validation or performance improvement is claimed by successful execution.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
REPO = Path('/content/healthbrowse-independent-model')
BRANCH = 'research/healthbrowse-medical-model'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/McChandra/medical_chatbot.git', str(REPO)], check=True)
if not (REPO / 'healthbrowse_model_lab/pyproject.toml').is_file():
    raise RuntimeError('This checkout does not contain the separate model lab. Use a fresh runtime and rerun this notebook.')
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Separate source version:', SOURCE_COMMIT)


In [ ]:
# Installs only this project's dependencies. Do not reuse the browser environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,test]'], check=True)


In [ ]:
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Open Colab's key icon, add a secret named HF_TOKEN, enable notebook access, then rerun this cell. Never paste the token into a code cell.") from exc
from hb_model_lab.training import preflight
CHECK = preflight('configs/llama3-8b-qlora.json')
print(json.dumps(CHECK, indent=2))
if not CHECK['cuda']:
    raise RuntimeError('No CUDA GPU is available. Select a GPU runtime before training.')


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Choose the run type

Leave `RUN_MODE = 'smoke'` for the first attempt; no uploads are needed. It checks that the real GPU can load the base model and save a new adapter.

Use `RUN_MODE = 'reviewed'` only after preparing the two JSONL files with valid rights and target-review records. The upload cell expects exactly `train.jsonl` and `validation.jsonl`. Their topic families and source pages must not overlap. The examples in this notebook are not a replacement for that curated dataset.


In [ ]:
RUN_MODE = 'smoke'  # 'smoke' or 'reviewed'
if RUN_MODE not in {'smoke', 'reviewed'}:
    raise ValueError('Choose smoke or reviewed')
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
RUN_ROOT = Path('/content/drive/MyDrive/HealthBrowseModelLab') / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
if RUN_MODE == 'smoke':
    TRAIN, VALIDATION = LAB / 'data/examples/train.jsonl', LAB / 'data/examples/validation.jsonl'
else:
    uploaded = files.upload()
    if set(uploaded) != {'train.jsonl', 'validation.jsonl'}:
        raise ValueError('Select exactly the reviewed train.jsonl and validation.jsonl files.')
    DATA = RUN_ROOT / 'data'
    DATA.mkdir()
    for filename, contents in uploaded.items():
        (DATA / filename).write_bytes(contents)
    del uploaded
    TRAIN, VALIDATION = DATA / 'train.jsonl', DATA / 'validation.jsonl'
from hb_model_lab.data import read_jsonl, validate_rows, validate_splits
tr = validate_rows(read_jsonl(TRAIN), allow_examples=RUN_MODE == 'smoke')
va = validate_rows(read_jsonl(VALIDATION), allow_examples=RUN_MODE == 'smoke')
validate_splits(tr, va)
print({'run': str(RUN_ROOT), 'train': len(tr), 'validation': len(va), 'mode': RUN_MODE})


In [ ]:
# Calls training directly so the actual error is visible, not only CalledProcessError.
from hb_model_lab.training import train
MANIFEST = train('configs/llama3-8b-qlora.json', TRAIN, VALIDATION,
                 RUN_ROOT / 'model', smoke=RUN_MODE == 'smoke')
print(json.dumps(MANIFEST, indent=2))
print('Saved separately:', RUN_ROOT / 'model/adapter')


## After training

A smoke run is only a software/hardware check; the research API rejects smoke weights. For a reviewed candidate, freeze an independently reviewed test set with no training leakage. Run the base and adapter evaluation under identical settings:

```python
from hb_model_lab.evaluation import evaluate
base = evaluate('configs/llama3-8b-qlora.json', 'evaluation/benchmark.jsonl',
                'data/education-corpus.jsonl.gz', RUN_ROOT / 'eval-base', baseline=True)
```

Each full evaluation runs 200 cases. Use a fresh runtime between baseline and adapter runs if necessary to release GPU memory; restore `RUN_ROOT` to the saved run directory before continuing. Then evaluate `adapter=str(RUN_ROOT / 'model/adapter')` to a new `eval-adapter` directory and compare results with the CLI. These cells are deliberately not auto-executed.

Inspect `predictions.jsonl` and complete independent human review. JSON validity, training loss and quote presence do not establish factual accuracy or clinical safety. Nothing here deploys to the live browser.
